<a href="https://colab.research.google.com/github/Fatima-05/FlyRank-ML/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fatima-05/FlyRank-ML/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [11]:
# My rule and its reason codes

# Lane: Refresh / Content Opportunity Scoring

# Rule in plain words:
# Score a page higher when it has real visibility and looks stale, declining, or weak on CTR. Prefer pages a small team would actually bother refreshing. Do not use future outcomes or product decision flags.

# Signals this rule leans on:
# 1. Content age / staleness (linked to FlyRank refresh-style flags)
# 2. Impressions volume (linked to “worth the effort” / quick-win logic)

# Reason codes:
# declining_with_demand
# stale_visible_page
# low_ctr_visible_page
# monitor

# Actions:
# refresh
# refresh_and_review_ctr
# monitor

import os
from pathlib import Path
import pandas as pd
import numpy as np

# Clone your repo if needed
if not Path("/content/FlyRank-ML").exists():
    %cd /content
    !git clone --depth 1 https://github.com/Fatima-05/FlyRank-ML

%cd /content/FlyRank-ML
print("CWD:", os.getcwd())
print("CSV exists:", Path("data/raw/content_refresh_anonymized.csv").exists())

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("Rows:", len(df))
print(df.columns.tolist()[:20])
df.head(3)

# signal 1: staleness
df["staleness_bucket"] = pd.cut(
    df["content_age_days"],
    bins=[-1, 90, 180, 365, 10_000],
    labels=["fresh", "aging", "stale", "very_stale"]
)
print("Signal 1: Staleness buckets:")
print(df["staleness_bucket"].value_counts(dropna=False))
print("n =", len(df))

# signal 2: impressions volume
df["imp_bucket"] = pd.cut(
    df["impressions_90d"],
    bins=[-1, 100, 500, 2000, 10_000_000],
    labels=["low", "mid", "high", "very_high"]
)
print("\nSignal 2: Impressions buckets:")
print(df["imp_bucket"].value_counts(dropna=False))
print("n =", len(df))

/content/FlyRank-ML
CWD: /content/FlyRank-ML
CSV exists: True
Rows: 30000
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d']
Signal 1: Staleness buckets:
staleness_bucket
aging         11780
stale         11368
very_stale     6360
fresh           492
Name: count, dtype: int64
n = 30000

Signal 2: Impressions buckets:
imp_bucket
very_high    10213
low           8006
high          6502
mid           5279
Name: count, dtype: int64
n = 30000


Signal checks

Signal 1 — Staleness (content_age_days)
Buckets (n = 30,000):
aging: 11,780
stale: 11,368
very_stale: 6,360
fresh: 492

Most pages are aging or stale. Very few are fresh.
Verdict: CONFIRMED
Staleness is a real, widespread signal in this dataset and is a fair input for a refresh rule.

Signal 2 — Impressions volume (impressions_90d)
Buckets (n = 30,000):
very_high: 10,213
high: 6,502
mid: 5,279
low: 8,006

There is clear spread across volume levels, so volume can separate worth-the-effort pages from low-reach ones.
Verdict: CONFIRMED
Impressions volume is a usable prioritization signal.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [12]:
# 2. Build the ranked queue

# One baseline rule only:
# numeric score
# one reason code
# one action label

# Output file: work/outputs/baseline_action_score.csv


def baseline_score(row):
    score = 0
    reason = "monitor"
    action = "monitor"

    trend = str(row.get("trend_direction", "")).lower()
    imp = row.get("impressions_90d", 0) or 0
    age = row.get("content_age_days", 0) or 0
    ctr = row.get("ctr", 1) or 1

    if imp >= 500 and trend == "down":
        score += 40
        reason = "declining_with_demand"
        action = "refresh"

    if age >= 180 and imp >= 250:
        score += 30
        if score >= 40:
            reason = "declining_with_demand"
        else:
            reason = "stale_visible_page"
        action = "refresh"

    if imp >= 500 and ctr < 0.02:
        score += 20
        reason = "low_ctr_visible_page"
        action = "refresh_and_review_ctr"

    return pd.Series({
        "baseline_score": score,
        "reason_code": reason,
        "action": action
    })

scored = df.join(df.apply(baseline_score, axis=1))
ranked = scored.sort_values("baseline_score", ascending=False).reset_index(drop=True)
ranked["rank"] = ranked.index + 1

out = Path("work/outputs")
out.mkdir(parents=True, exist_ok=True)
ranked.head(200).to_csv(out / "baseline_action_score.csv", index=False)

print("Wrote work/outputs/baseline_action_score.csv")
print(ranked[["rank", "content_id", "baseline_score", "reason_code", "action", "impressions_90d", "content_age_days"]].head(10))

Wrote work/outputs/baseline_action_score.csv
   rank            content_id  baseline_score           reason_code  \
0     1  content_5a46cb402872              90  low_ctr_visible_page   
1     2  content_a38dd531fd8f              90  low_ctr_visible_page   
2     3  content_b92a65abeeab              90  low_ctr_visible_page   
3     4  content_124763d39ca5              90  low_ctr_visible_page   
4     5  content_0b18afd5b14b              90  low_ctr_visible_page   
5     6  content_36684976be1f              90  low_ctr_visible_page   
6     7  content_e6642d0fa24d              90  low_ctr_visible_page   
7     8  content_94ff4ec6b6b5              90  low_ctr_visible_page   
8     9  content_e348b1dda5aa              90  low_ctr_visible_page   
9    10  content_971e2a5035bc              90  low_ctr_visible_page   

                   action  impressions_90d  content_age_days  
0  refresh_and_review_ctr            10305               557  
1  refresh_and_review_ctr            22716     

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [13]:
# 3. Top-20 review

# For each top row: action, reason code, short confidence note, and what would make it wrong.

top20 = ranked.head(20)[
    ["rank", "content_id", "baseline_score", "reason_code", "action",
     "impressions_90d", "content_age_days", "trend_direction", "ctr"]
]
top20

,rank,content_id,baseline_score,reason_code,action,impressions_90d,content_age_days,trend_direction,ctr
0,1,content_5a46cb402872,90,low_ctr_visible_page,refresh_and_review_ctr,10305,557,down,0.01
1,2,content_a38dd531fd8f,90,low_ctr_visible_page,refresh_and_review_ctr,22716,236,down,0.01
2,3,content_b92a65abeeab,90,low_ctr_visible_page,refresh_and_review_ctr,18837,445,down,0.01
3,4,content_124763d39ca5,90,low_ctr_visible_page,refresh_and_review_ctr,129803,286,down,0.01
4,5,content_0b18afd5b14b,90,low_ctr_visible_page,refresh_and_review_ctr,7208,300,down,0.01
5,6,content_36684976be1f,90,low_ctr_visible_page,refresh_and_review_ctr,9844,445,down,0.01
6,7,content_e6642d0fa24d,90,low_ctr_visible_page,refresh_and_review_ctr,8085,307,down,0.01
7,8,content_94ff4ec6b6b5,90,low_ctr_visible_page,refresh_and_review_ctr,14658,230,down,0.01
8,9,content_e348b1dda5aa,90,low_ctr_visible_page,refresh_and_review_ctr,10392,300,down,0.01
9,10,content_971e2a5035bc,90,low_ctr_visible_page,refresh_and_review_ctr,9179,287,down,0.01


3. Top-20 review

All of the current top 20 scored 90 with reason low_ctr_visible_page and action refresh_and_review_ctr.
They share the same pattern: high impressions, trend = down, CTR around 0.01, and age usually 200+ days.

1. content_5a46cb402872: refresh_and_review_ctr. High impressions + low CTR + declining. Wrong if CTR is normal for this position/intent.
2. content_a38dd531fd8f: refresh_and_review_ctr. Very high impressions + low CTR + declining. Wrong if the page is intentionally non-click (brand/navigational).
3. content_b92a65abeeab: refresh_and_review_ctr. High impressions + stale + low CTR. Wrong if content is already recently rewritten offline.
4. content_124763d39ca5: refresh_and_review_ctr. Extremely high impressions + low CTR. Wrong if ranking is already strong and CTR is expected for the SERP.
5. content_0b18afd5b14b: refresh_and_review_ctr. Visible + aging + low CTR. Wrong if traffic is seasonal and currently off-peak.
6. content_36684976be1f: refresh_and_review_ctr. High impressions + stale + declining. Wrong if decline is caused by site-wide issues, not this page.
7. content_e6642d0fa24d: refresh_and_review_ctr. Solid visibility + low CTR. Wrong if title/meta were just changed and need time.
8. content_94ff4ec6b6b5: refresh_and_review_ctr. High impressions + declining. Wrong if competing pages should be consolidated instead.
9. content_e348b1dda5aa: refresh_and_review_ctr. Visible + aging + low CTR. Wrong if the page is a thin supporting URL, not a refresh target.
10. content_971e2a5035bc: refresh_and_review_ctr. High impressions + low CTR. Wrong if measurement noise makes CTR look artificially low.
11. content_54baba704595: refresh_and_review_ctr. Very high impressions + declining. Wrong if this is a hub page that should be monitored, not rewritten.
12. content_5096a9d25fe5: refresh_and_review_ctr. High impressions + low CTR. Wrong if intent mismatch needs a different page, not a refresh.
13. content_94058fab0b5b: refresh_and_review_ctr. High impressions + aging. Wrong if decline is from algorithm volatility, not content quality.
14. content_b82f592415bc: refresh_and_review_ctr. Visible + stale-ish + low CTR. Wrong if page is already scheduled in an editorial calendar.
15. content_c2cca3b2a2ef: refresh_and_review_ctr. High impressions + declining. Wrong if cannibalization is the real issue.
16. content_33a2ec3db00c: refresh_and_review_ctr. High impressions + aging. Wrong if newer internal links would fix it without a full refresh.
17. content_058efde65398: refresh_and_review_ctr. Solid visibility + low CTR. Wrong if SERP features suppress CTR for everyone.
18. content_5feee3994adb: refresh_and_review_ctr. Visible + declining. Wrong if the page is near deletion/merge already.
19. content_a01545948bb5: refresh_and_review_ctr. Visible + low CTR. Wrong if brand-query behavior makes low CTR expected.
20. content_7713c488c8e8: refresh_and_review_ctr. High impressions + declining. Wrong if a technical issue is blocking clicks.

Pattern note: The top of this baseline is dominated by one reason code. That is useful as a simple rule, but also a weakness: the queue is less diverse than a fuller model would be.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [14]:
# 4. Weak picks + leakage check

# Weak picks: pages that scored high but may not deserve action (seasonal pages, already strong pages, thin evidence).

# Leakage check:
# No future-window labels used as features
# No product flags / health_score / priority_score used
# Only observable past signals: impressions, age, trend_direction, ctr


used_cols = ["impressions_90d", "content_age_days", "trend_direction", "ctr"]
print("Columns used by rule:", used_cols)
print("Any obviously future/label-only columns used? No.")
print(ranked["action"].value_counts())

Columns used by rule: ['impressions_90d', 'content_age_days', 'trend_direction', 'ctr']
Any obviously future/label-only columns used? No.
action
refresh                   16071
monitor                   13851
refresh_and_review_ctr       78
Name: count, dtype: int64


4. Weak picks + leakage check

Weak picks / limitations
Top ranks are almost all low_ctr_visible_page. The rule may over-weight the CTR < 0.02 threshold.
High-impression declining pages can look urgent even when the real fix is technical, seasonal, or cannibalization.
Pages with CTR that is low in absolute terms may still be normal for their position or intent.
A score of 90 for many rows means the rule does not separate the very top finely enough.

Leakage check
Columns used: impressions_90d, content_age_days, trend_direction, ctr
No product flags, health scores, or priority scores used
No future-window labels used as features
This is a same-window baseline for ranking practice, not a claim of causal refresh impact

Action mix in full scored set
refresh: 16,071
monitor: 13,851
refresh_and_review_ctr: 78

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.